# The null controls, and what survives them

* **White noise**: what spectra of pure noise look like - peaks included.
* **Detrended random walk**: rolling detrending shapes a spectrum by itself; this is the control for that.
* **Shuffled returns**: the real return distribution with all serial dependence removed.
* **Block bootstrap**: short-range dependence and volatility clustering kept inside blocks, long-range structure removed.

The research ledger records every hypothesis tested, including the ones that failed.

**This notebook contains no calculations.** Everything is computed by `xauusd_quant.features.spectral` and `xauusd_quant.research` and read back from `results/spectral_research/<tf>/<series>/fft_<N>/`. Generate those first:

```
xq spectral-research --timeframe 5m --source regression_residual --fft-window 256
xq spectral-research --timeframe 5m --all-windows
xq spectral-research --all
```

Every number sits beside four null controls: **white noise**, a **detrended random walk** (the same regression and OU pipeline on a series with no structure), and **shuffled** and **block-bootstrapped** real returns. A spectral peak is not a cycle until it survives them.

No trading rule is defined or evaluated anywhere. No costs are applied. Nothing here is a signal.

In [ ]:
import sys
import json
from pathlib import Path

# Work against the repository's own source tree, not an installed copy.
ROOT = Path.cwd().parent if Path.cwd().name == "research" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import polars as pl
from IPython.display import Image, display

from xauusd_quant.features.spectral_config import load_spectral_config

spectral = load_spectral_config(ROOT / "config" / "spectral.yaml")

TIMEFRAME = "5m"                    # change and re-run
SERIES = "regression_residual"      # regression_residual | ou_innovation | log_return
FFT_WINDOW = 256                    # N bars per window

results = spectral.results_dir(TIMEFRAME, SERIES, FFT_WINDOW)
if not results.exists():
    raise SystemExit(
        f"No results for {TIMEFRAME} {SERIES} N={FFT_WINDOW}. Run:  xq spectral-research "
        f"--timeframe {TIMEFRAME} --source {SERIES} --fft-window {FFT_WINDOW}")

def table(name):
    """Load a previously generated spectral table (None if it was not written)."""
    path = results / f"{name}.parquet"
    return pl.read_parquet(path) if path.exists() else None

def plot(name):
    path = results / "plots" / f"{name}.png"
    if path.exists():
        display(Image(filename=str(path)))

payload = json.loads((results / "summary.json").read_text(encoding="utf-8"))
summary = payload["summary"]
lineage = payload["provenance"]["dataset"]
print(f"bars       : {summary['observations']:,} ({summary['valid_spectral_fraction']:.1%} "
      f"with a valid {FFT_WINDOW}-bar window)")
print(f"ticks      : {lineage['tick_dataset_version']} ({lineage['tick_dataset_status']})")
print(f"bars       : {lineage.get('bar_dataset_version')}")
if lineage.get("label"):
    print(f"WARNING    : {lineage['label']}")
for warning in payload.get("warnings", []):
    print(f"WARNING    : {warning}")

In [ ]:
table('null_control_comparison')

## Residual decay by spectral bucket, real beside the nulls

In [ ]:
table('residual_outcomes')

## Information coefficients - how many were tested, and how they compare

In [ ]:
ic = table('ic_analysis')
print("tests per source:", summary["ic_tests"])
ic.sort(pl.col("rank_ic").abs(), descending=True).head(20)

## Redundancy with existing regression, OU and volatility features

In [ ]:
table('feature_redundancy')

## The research ledger: every hypothesis and its verdict

In [ ]:
ledger_path = spectral.ledger_path
ledger = pl.read_parquet(ledger_path) if ledger_path.exists() else None
(ledger.filter((pl.col("timeframe") == TIMEFRAME) & (pl.col("input_series") == SERIES)
               & (pl.col("fft_window") == FFT_WINDOW))
 .group_by("hypothesis_id", "verdict").len().sort("hypothesis_id")) if ledger is not None else None

## Across every study: how many tests, and how often chance alone exceeds the nulls

`hypothesis_test_counts` is the multiple-testing record: every hypothesis, how many times it was tested and with what verdicts. `ic_chance_rates` puts the real series' IC exceedances beside each null's own count against the other sources - the rate chance produces. Both are written by `--all` and `--compare`.

In [ ]:
counts = spectral.results_path / "hypothesis_test_counts.csv"
pl.read_csv(counts) if counts.exists() else "run xq spectral-research --compare"

In [ ]:
rates = spectral.results_path / "ic_chance_rates.csv"
(pl.read_csv(rates).group_by("input_series", "source")
 .agg(pl.col("tests").sum(), pl.col("exceedances").sum(),
      pl.col("exceedances_stable_sign").sum())
 .sort("input_series", "source")) if rates.exists() else "run xq spectral-research --compare"